# S07 - Word Embeddings & Neural Networks
## Exercises

### Exercise 1 (Easy)
Load pre-trained Word2Vec embeddings and find similar words.

In [2]:
import gensim.downloader as api

# Load pre-trained word2vec (google-news-300 or glove-wiki-gigaword-100)
# Find top 5 most similar words to 'king'

model = api.load('glove-wiki-gigaword-100')
similar = model.most_similar('king', topn=5)
print("Similar to 'king':")
for word, score in similar:
    print(f"  {word}: {score:.3f}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Similar to 'king':
  prince: 0.768
  queen: 0.751
  son: 0.702
  brother: 0.699
  monarch: 0.698


### Exercise 2 (Easy)
Perform word analogy: king - man + woman = ?

In [3]:
# Use the model to solve: king - man + woman = ?
# Also try: paris - france + spain = ?

# king - man + woman = queen
result = model.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)
print(f"king - man + woman = {result[0][0]}")

# paris - france + spain = madrid
result = model.most_similar(positive=['paris', 'spain'], negative=['france'], topn=1)
print(f"paris - france + spain = {result[0][0]}")

king - man + woman = queen
paris - france + spain = madrid


### Exercise 3 (Medium)
Train your own Word2Vec model on a custom corpus.

In [4]:
from gensim.models import Word2Vec

corpus = [
    ["the", "cat", "sat", "on", "the", "mat"],
    ["the", "dog", "ran", "in", "the", "park"],
    ["cats", "and", "dogs", "are", "pets"],
    ["the", "cat", "chased", "the", "dog"],
    ["pets", "need", "food", "and", "water"]
]

# Train Word2Vec model (vector_size=50, window=3, min_count=1)

model = Word2Vec(corpus, vector_size=50, window=3, min_count=1, epochs=100)

print("Vocabulary:", list(model.wv.key_to_index.keys()))
print(f"\nVector for 'cat': {model.wv['cat'][:5]}...")
print(f"\nSimilar to 'cat': {model.wv.most_similar('cat', topn=3)}")


Vocabulary: ['the', 'pets', 'and', 'dog', 'cat', 'water', 'food', 'need', 'chased', 'are', 'dogs', 'cats', 'park', 'in', 'ran', 'mat', 'on', 'sat']

Vector for 'cat': [ 3.4071356e-05  6.1588753e-03 -1.3525375e-02 -2.5881715e-03
  1.5226374e-02]...

Similar to 'cat': [('and', 0.180289626121521), ('in', 0.16276536881923676), ('need', 0.14904862642288208)]


### Exercise 4 (Medium)
Build a simple neural network for text classification using embeddings.

In [5]:
import torch
import torch.nn as nn

class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, num_classes)
    
    def forward(self, x):
        embedded = self.embedding(x)  # (batch, seq_len, embed_dim)
        pooled = embedded.mean(dim=1)  # (batch, embed_dim)
        return self.fc(pooled)

# Test with dummy data

model = TextClassifier(vocab_size=1000, embed_dim=100, num_classes=2)
x = torch.randint(0, 1000, (4, 10))  # batch=4, seq_len=10
out = model(x)
print(f"Output shape: {out.shape}")  # (4, 2)

Output shape: torch.Size([4, 2])


### Exercise 5 (Hard)
Implement the Skip-gram model from scratch (forward pass only).

*Research: Skip-gram predicts context words given center word.*

In [6]:
import torch
import torch.nn as nn

class SkipGram(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.center_embeddings = nn.Embedding(vocab_size, embed_dim)
        self.context_embeddings = nn.Embedding(vocab_size, embed_dim)
    
    def forward(self, center, context):
        # center: (batch,), context: (batch,)
        center_emb = self.center_embeddings(center)  # (batch, embed_dim)
        context_emb = self.context_embeddings(context)  # (batch, embed_dim)
        # Dot product for similarity score
        scores = (center_emb * context_emb).sum(dim=1)
        return scores

# Test
model = SkipGram(vocab_size=100, embed_dim=50)
center = torch.tensor([5, 10, 15])
context = torch.tensor([6, 11, 16])
scores = model(center, context)
print(f"Scores: {scores}")

Scores: tensor([ 0.2483,  4.1880, -0.2246], grad_fn=<SumBackward1>)
